In [ ]:
# from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pylab as pl
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import collections as mc
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

plt.style.use("ggplot")
plt.style.use("seaborn-v0_8-colorblind")

In [ ]:
# torch.load('model_checkpoint/model.pth')

In [ ]:
# load data
df = pd.read_csv(
    "./dataset/Dataset_VisContest_Rapid_Alloy_development_v3.txt",
    encoding="latin1",
    sep="\t",
)
display(df.head(2))
print("Total number of rows", len(df))

In [ ]:
# partition data by categories
n_inputs = 6
n_elements = 12

all_cols = df.columns.tolist()
input_cols = all_cols[:n_inputs]
element_cols = all_cols[n_inputs : n_inputs + n_elements]
microstructure_cols = all_cols[18 : 18 + 38]
property_cols = all_cols[56:70]
print(f"{len(input_cols)} Input scraps:", input_cols, end="\n\n")
print(f"{len(element_cols)} Elements:", element_cols, end="\n\n")
print(f"{len(microstructure_cols)} Microstructures:", microstructure_cols, end="\n\n")
print(f"{len(property_cols)} Properties:", property_cols, end="\n\n")

In [ ]:
# handle na
df = df.fillna(value=0)

# # # subsample
# n = 20000  # number of samples
# df = df.sample(n)  # sample data for speed&test
# # df.to_csv(f'./dataset/data_sample_{n}.csv', index=False)
# # df

In [ ]:
# Create training data
x_cols = element_cols
y_cols = property_cols

X = df[x_cols].to_numpy().astype(np.float32)
y = df[y_cols].to_numpy().astype(np.float32)

# normalize
y_mean = y.mean(axis=0)
y_std = y.std(axis=0)
y = (y - y_mean) / (y_std)

X, y = torch.from_numpy(X), torch.from_numpy(y)

In [ ]:
print(f"{'property':>50} | {'mean':>15}  {'std':>15}")
print("-" * 120)
for name, mean, std in zip(y_cols, y_mean, y_std):
    print(f"{name:>50} : {mean:15.4f}, {std:15.4f}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device(
        "mps"
    )  # Metal Performance Shaders (MPS) for Apple Silicon Macs
else:
    device = torch.device("cpu")

print("Using device:", device)


X, y = X.to(device), y.to(device)

In [ ]:
# model.load_state_dict(torch.load("model_checkpoint/model_best.pth"))

In [ ]:
def save_model(model, y_mean, y_std, x_cols, y_cols, path="model_checkpoint/model.pth"):
    torch.save(
        dict(
            state_dict=model.state_dict(),
            y_mean=torch.from_numpy(y_mean),
            y_std=torch.from_numpy(y_std),
            x_cols=x_cols,
            y_cols=y_cols,
        ),
    )

In [ ]:
# Define a simple MLP
class SurrogatePropertyPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            # nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.PReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.PReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


def max_residual(pred, truth):
    return np.abs(pred - truth).max(axis=0)

In [ ]:
# Hyperparameters
hidden_dim = 1024
batch_size = 512
epochs = 200
lr = 0.001

input_dim = X.shape[1]
output_dim = y.shape[1]

# DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Model, loss, optimizer
model = SurrogatePropertyPredictor(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
# optimizer = optim.SGD(model.parameters(), lr=lr)

# training prep
rcurrent = 1e4
test_sample = torch.randperm(y.size(0))[:3000]
pbar = tqdm(range(epochs))

In [ ]:
model_checkpoint = torch.load("model_checkpoint/best.pth", weights_only=True)
model.load_state_dict(model_checkpoint["state_dict"])

In [ ]:
# # Training loop
# for epoch in pbar:
#     for batch_X, batch_y in loader:
#         optimizer.zero_grad()
#         output = model(batch_X)
#         loss = criterion(output, batch_y)
#         loss.backward()
#         optimizer.step()

#     if epoch % 10 == 9:
#         pbar.set_postfix_str(f"Epoch {epoch+1}, Loss: {loss.item():.8f}")

#         pred = model(X[test_sample]).detach().cpu().numpy()
#         truth = y[test_sample].cpu().numpy()

#         residuals = max_residual(pred, truth)
#         rmax = residuals.max()
#         if rmax < rcurrent:
#             # torch.save(model.state_dict(), "model_checkpoint/model_best.pth")
#             save_model(
#                 model,
#                 y_mean,
#                 y_std,
#                 x_cols,
#                 y_cols,
#                 path="model_checkpoint/best.pth",
#             )
#             rcurrent = rmax

#         print(f"{'property':>50} | Max residuals")
#         for name, res in zip(y_cols, residuals):
#             print(f"{name:>50} : {res}")
#         print()

## Performance Vis

In [ ]:
sample = torch.randperm(y.size(0))[:10000]
pred = model(X[sample]).detach().cpu()
truth = y[sample].cpu()

vmax = max(truth.max(), pred.max())
plt.scatter(truth.flatten(), pred.flatten(), zorder=2)
plt.plot([0, vmax], [0, vmax], c="#aaa", zorder=1, linewidth=0.5)
plt.xlabel("Ground truth")
plt.ylabel("Prediction")

# plot pred, ground truth and residuals against X

lines = list(
    zip(
        zip(X[sample, 0].cpu().numpy(), truth[:, 0].cpu().numpy()),
        zip(X[sample, 0].cpu().numpy(), pred[:, 0].detach().cpu().numpy()),
    )
)
lc = mc.LineCollection(lines, linewidths=0.5, colors="#aaa", zorder=1)
fig, ax = pl.subplots()
ax.add_collection(lc)
ax.autoscale()
ax.margins(0.1)
plt.scatter(
    X[sample, 0].cpu().numpy(), truth[:, 0].cpu().numpy(), s=5, label="ground truth"
)
plt.scatter(
    X[sample, 0].cpu().numpy(),
    pred[:, 0].detach().cpu().numpy(),
    s=5,
    c="C2",
    label="pred",
)
plt.legend()
plt.show()

## Test UI

In [ ]:
from random import randint

In [ ]:
sample = torch.randperm(y.size(0))[:10000]
# x_ref = X.mean(axis=0)
x_ref = X[randint(0, X.shape[0])]
plt.figure(figsize=[24, 24])

for xdim in range(0, len(x_cols)):
    for ydim in range(0, len(y_cols)):

        r = torch.arange(len(x_cols))
        all_except_current = r != xdim

        close_to_ref = (
            (X[:, all_except_current] - x_ref[all_except_current])
            .abs()
            .max(axis=1)
            .values
            < 0.2
        ).cpu()
        # print(close_to_ref.sum().item())
        # close_to_ref

        plt.subplot(len(y_cols), len(x_cols), ydim * len(x_cols) + xdim + 1)
        
        plt.scatter(
            X[sample, xdim].cpu().numpy(),
            y[sample, ydim].cpu().numpy(),
            s=5,
            c="#333",
        )
        plt.scatter(
            X.cpu()[close_to_ref, xdim].numpy(),
            y.cpu()[close_to_ref, ydim].numpy(),
            s=5,
            label="ground truth",
        )

        n = 100
        x_curve = x_ref.repeat([n, 1])
        x_curve[:, xdim] = torch.linspace(X[:, xdim].min(), X[:, xdim].max(), n)
        # x_curve[:, xdim] = torch.linspace(
        #     X[close_to_ref, xdim].min(), X[close_to_ref, xdim].max(), n
        # )
        with torch.no_grad():
            y_curve = model(x_curve)
        plt.plot(
            x_curve[:, xdim].cpu().numpy(), y_curve[:, ydim].cpu().numpy(), c="orange"
        )
        if ydim == len(y_cols)-1:
            plt.xlabel(x_cols[xdim])
        if xdim == 0:
            plt.ylabel(y_cols[ydim])
plt.show()

In [ ]:
all_except_current